# 15b — Clustering Method Comparison: K-Means vs GMM vs HDBSCAN

**Project:** MARS — Multi-Agent Recommender System for Personalized Learning  
**Prompt:** 3.2 — Learner clustering method comparison  
**Purpose:** Evaluate three clustering algorithms on EdNet KT2 learner feature space
to justify the K-Means default used in PersonalizationAgent.

| Method | Selection Criterion | Hyperparameters |
|--------|--------------------|-----------------|
| K-Means | Silhouette Score | K ∈ [3, 8] |
| GMM | BIC (lower is better) | K ∈ [3, 8] |
| HDBSCAN | Automatic | min_cluster_size ∈ {10, 20, 50}, min_samples ∈ {5, 10} |

**Metrics:** Silhouette, Calinski-Harabasz, Davies-Bouldin, Cluster Stability (ARI), Outlier Fraction

In [ ]:
"""Cell 1 — Imports and setup."""

from __future__ import annotations

import json
import logging
import sys
import warnings
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

try:
    import hdbscan as _hdbscan_lib
    HDBSCAN = _hdbscan_lib.HDBSCAN
    print("Using hdbscan library")
except ImportError:
    from sklearn.cluster import HDBSCAN
    print("Using sklearn HDBSCAN")

warnings.filterwarnings("ignore")

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from agents.utils import set_global_seed, load_config
from agents.personalization_agent import PersonalizationAgent, extract_user_features
from data.loader import EdNetLoader

plt.style.use("seaborn-v0_8-paper")
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300,
    "font.size": 11, "axes.titlesize": 13,
    "axes.labelsize": 12, "figure.figsize": (8, 5),
    "savefig.bbox": "tight",
})

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("clustering_comparison")
logger.setLevel(logging.INFO)

SEEDS = [42, 123, 456, 789, 2024]
K_RANGE = range(3, 9)  # K in [3, 8]
HDBSCAN_MIN_CLUSTER_SIZES = [10, 20, 50]
HDBSCAN_MIN_SAMPLES = [5, 10]

RESULTS_DIR = ROOT / "results" / "clustering_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = load_config(str(ROOT / "configs" / "config.yaml"))

print(f"K range: {list(K_RANGE)}")
print(f"Seeds: {SEEDS}")
print(f"Results dir: {RESULTS_DIR}")

## 1. Load Data & Extract Features

In [ ]:
"""Cell 2 — Load training data and extract user features."""

splits_dir = ROOT / "data" / "splits"
train_df = pd.read_parquet(splits_dir / "train.parquet")

print(f"Train interactions: {len(train_df):,}")
print(f"Train users: {train_df['user_id'].nunique():,}")

# Extract user features (14 features per user):
# 7 scalar: avg_elapsed_time, accuracy_rate, changed_answer_rate,
#           session_frequency, avg_questions_per_session, false_confidence_rate, learning_speed
# 7 one-hot: dominant_part_1 to dominant_part_7
user_feats = extract_user_features(train_df)
print(f"\nUser features shape: {user_feats.shape}")
print(f"Columns: {list(user_feats.columns)}")

# StandardScaler before clustering
feature_cols = [c for c in user_feats.columns if c != "user_id"]
scaler = StandardScaler()
X = scaler.fit_transform(
    np.nan_to_num(user_feats[feature_cols].values, nan=0.0)
)

print(f"Scaled feature matrix: {X.shape}")
print(f"Feature names: {feature_cols}")

## 2. K-Means Clustering

In [ ]:
"""Cell 3 — K-Means: sweep K in [3,8], select by silhouette."""

kmeans_results = {}

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(X)

    sil = silhouette_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    db = davies_bouldin_score(X, labels)

    kmeans_results[k] = {
        "labels": labels,
        "model": km,
        "silhouette": sil,
        "calinski_harabasz": ch,
        "davies_bouldin": db,
        "inertia": km.inertia_,
        "n_clusters": k,
    }
    print(f"K={k}: Silhouette={sil:.4f}, CH={ch:.1f}, DB={db:.4f}")

best_k_kmeans = max(kmeans_results, key=lambda k: kmeans_results[k]["silhouette"])
print(f"\nBest K (by Silhouette): {best_k_kmeans} "
      f"(Sil={kmeans_results[best_k_kmeans]['silhouette']:.4f})")

## 3. GMM Clustering

In [ ]:
"""Cell 4 — GMM: sweep K in [3,8], select by BIC."""

gmm_results = {}

for k in K_RANGE:
    gmm = GaussianMixture(
        n_components=k, covariance_type="full",
        random_state=42, n_init=5, max_iter=300,
    )
    gmm.fit(X)
    labels = gmm.predict(X)

    sil = silhouette_score(X, labels)
    ch = calinski_harabasz_score(X, labels)
    db = davies_bouldin_score(X, labels)
    bic = gmm.bic(X)
    aic = gmm.aic(X)

    gmm_results[k] = {
        "labels": labels,
        "model": gmm,
        "silhouette": sil,
        "calinski_harabasz": ch,
        "davies_bouldin": db,
        "bic": bic,
        "aic": aic,
        "n_clusters": k,
    }
    print(f"K={k}: BIC={bic:.1f}, AIC={aic:.1f}, "
          f"Sil={sil:.4f}, CH={ch:.1f}, DB={db:.4f}")

best_k_gmm = min(gmm_results, key=lambda k: gmm_results[k]["bic"])
print(f"\nBest K (by BIC): {best_k_gmm} "
      f"(BIC={gmm_results[best_k_gmm]['bic']:.1f}, "
      f"Sil={gmm_results[best_k_gmm]['silhouette']:.4f})")

## 4. HDBSCAN Clustering

In [ ]:
"""Cell 5 — HDBSCAN: grid search min_cluster_size x min_samples."""

hdbscan_results = {}

for mcs, ms in product(HDBSCAN_MIN_CLUSTER_SIZES, HDBSCAN_MIN_SAMPLES):
    key = f"mcs{mcs}_ms{ms}"
    clusterer = HDBSCAN(min_cluster_size=mcs, min_samples=ms)
    labels = clusterer.fit_predict(X)

    n_clusters = len(set(labels) - {-1})
    n_outliers = (labels == -1).sum()
    outlier_frac = n_outliers / len(labels)

    # Metrics only on non-outlier points
    mask = labels != -1
    if n_clusters >= 2 and mask.sum() > n_clusters:
        sil = silhouette_score(X[mask], labels[mask])
        ch = calinski_harabasz_score(X[mask], labels[mask])
        db = davies_bouldin_score(X[mask], labels[mask])
    else:
        sil, ch, db = -1.0, 0.0, float("inf")

    hdbscan_results[key] = {
        "labels": labels,
        "model": clusterer,
        "silhouette": sil,
        "calinski_harabasz": ch,
        "davies_bouldin": db,
        "n_clusters": n_clusters,
        "n_outliers": int(n_outliers),
        "outlier_fraction": outlier_frac,
        "min_cluster_size": mcs,
        "min_samples": ms,
    }
    print(f"mcs={mcs:3d}, ms={ms:2d}: "
          f"clusters={n_clusters}, outliers={n_outliers} ({outlier_frac:.1%}), "
          f"Sil={sil:.4f}, CH={ch:.1f}, DB={db:.4f}")

# Select best HDBSCAN config by silhouette (among configs with >= 2 clusters)
valid_hdb = {k: v for k, v in hdbscan_results.items()
             if v["n_clusters"] >= 2 and v["silhouette"] > -1}
if valid_hdb:
    best_hdb_key = max(valid_hdb, key=lambda k: valid_hdb[k]["silhouette"])
    bh = hdbscan_results[best_hdb_key]
    print(f"\nBest HDBSCAN: {best_hdb_key} "
          f"(clusters={bh['n_clusters']}, outliers={bh['outlier_fraction']:.1%}, "
          f"Sil={bh['silhouette']:.4f})")
else:
    best_hdb_key = list(hdbscan_results.keys())[0]
    print("\nWarning: no valid HDBSCAN config found with >= 2 clusters")

## 5. Cluster Stability Analysis (ARI across seeds)

In [ ]:
"""Cell 6 — Cluster stability: ARI between runs with different seeds."""

def compute_stability(method, X, seeds, **kwargs):
    """
    Run clustering with each seed, compute pairwise ARI between all
    seed pairs. Return mean and std ARI.
    """
    all_labels = []
    for seed in seeds:
        if method == "kmeans":
            model = KMeans(n_clusters=kwargs["k"], random_state=seed,
                           n_init=10, max_iter=300)
            labels = model.fit_predict(X)
        elif method == "gmm":
            model = GaussianMixture(
                n_components=kwargs["k"], covariance_type="full",
                random_state=seed, n_init=5, max_iter=300,
            )
            labels = model.fit_predict(X)
        elif method == "hdbscan":
            # HDBSCAN is deterministic given data, but we subsample
            rng = np.random.RandomState(seed)
            n = len(X)
            idx = rng.choice(n, size=int(0.9 * n), replace=False)
            idx_sorted = np.sort(idx)
            clusterer = HDBSCAN(
                min_cluster_size=kwargs["min_cluster_size"],
                min_samples=kwargs["min_samples"],
            )
            sub_labels = clusterer.fit_predict(X[idx_sorted])
            # Map back: assign non-sampled to -1
            full_labels = np.full(n, -1, dtype=int)
            full_labels[idx_sorted] = sub_labels
            labels = full_labels
        else:
            raise ValueError(f"Unknown method: {method}")
        all_labels.append(labels)

    # Pairwise ARI
    ari_scores = []
    for i in range(len(seeds)):
        for j in range(i + 1, len(seeds)):
            # For HDBSCAN: compute ARI only on points present in both subsamples
            if method == "hdbscan":
                mask = (all_labels[i] != -1) & (all_labels[j] != -1)
                if mask.sum() < 10:
                    continue
                ari = adjusted_rand_score(all_labels[i][mask], all_labels[j][mask])
            else:
                ari = adjusted_rand_score(all_labels[i], all_labels[j])
            ari_scores.append(ari)

    return np.mean(ari_scores), np.std(ari_scores), ari_scores


# K-Means stability
km_ari_mean, km_ari_std, km_aris = compute_stability(
    "kmeans", X, SEEDS, k=best_k_kmeans
)
print(f"K-Means (K={best_k_kmeans}) stability: "
      f"ARI = {km_ari_mean:.4f} +/- {km_ari_std:.4f}")

# GMM stability
gmm_ari_mean, gmm_ari_std, gmm_aris = compute_stability(
    "gmm", X, SEEDS, k=best_k_gmm
)
print(f"GMM (K={best_k_gmm}) stability: "
      f"ARI = {gmm_ari_mean:.4f} +/- {gmm_ari_std:.4f}")

# HDBSCAN stability
best_hdb = hdbscan_results[best_hdb_key]
hdb_ari_mean, hdb_ari_std, hdb_aris = compute_stability(
    "hdbscan", X, SEEDS,
    min_cluster_size=best_hdb["min_cluster_size"],
    min_samples=best_hdb["min_samples"],
)
print(f"HDBSCAN (mcs={best_hdb['min_cluster_size']}, ms={best_hdb['min_samples']}) "
      f"stability: ARI = {hdb_ari_mean:.4f} +/- {hdb_ari_std:.4f}")

## 6. Metric Comparison Summary Table

In [ ]:
"""Cell 7 — Build comparison table across all three methods."""

km_best = kmeans_results[best_k_kmeans]
gmm_best = gmm_results[best_k_gmm]
hdb_best = hdbscan_results[best_hdb_key]

comparison_rows = [
    {
        "Method": "K-Means",
        "K / Config": f"K={best_k_kmeans}",
        "Silhouette": km_best["silhouette"],
        "Calinski-Harabasz": km_best["calinski_harabasz"],
        "Davies-Bouldin": km_best["davies_bouldin"],
        "Stability (ARI)": f"{km_ari_mean:.4f} +/- {km_ari_std:.4f}",
        "Clusters Found": best_k_kmeans,
        "Outlier %": "0.0%",
    },
    {
        "Method": "GMM",
        "K / Config": f"K={best_k_gmm}",
        "Silhouette": gmm_best["silhouette"],
        "Calinski-Harabasz": gmm_best["calinski_harabasz"],
        "Davies-Bouldin": gmm_best["davies_bouldin"],
        "Stability (ARI)": f"{gmm_ari_mean:.4f} +/- {gmm_ari_std:.4f}",
        "Clusters Found": best_k_gmm,
        "Outlier %": "0.0%",
    },
    {
        "Method": "HDBSCAN",
        "K / Config": f"mcs={hdb_best['min_cluster_size']}, ms={hdb_best['min_samples']}",
        "Silhouette": hdb_best["silhouette"],
        "Calinski-Harabasz": hdb_best["calinski_harabasz"],
        "Davies-Bouldin": hdb_best["davies_bouldin"],
        "Stability (ARI)": f"{hdb_ari_mean:.4f} +/- {hdb_ari_std:.4f}",
        "Clusters Found": hdb_best["n_clusters"],
        "Outlier %": f"{hdb_best['outlier_fraction']:.1%}",
    },
]

comparison_df = pd.DataFrame(comparison_rows)
print("=" * 110)
print("Clustering Method Comparison")
print("=" * 110)
display(comparison_df)

comparison_df.to_csv(RESULTS_DIR / "clustering_comparison.csv", index=False)
print(f"\nSaved: {RESULTS_DIR / 'clustering_comparison.csv'}")

## 7. LaTeX Table for Paper

In [ ]:
"""Cell 8 — Generate LaTeX table."""

# Build formatted DataFrame for LaTeX
latex_rows = []
for row in comparison_rows:
    latex_rows.append({
        "Method": row["Method"],
        "Config": row["K / Config"],
        "Silhouette $\\uparrow$": f"{row['Silhouette']:.4f}",
        "CH $\\uparrow$": f"{row['Calinski-Harabasz']:.1f}",
        "DB $\\downarrow$": f"{row['Davies-Bouldin']:.4f}",
        "ARI Stability": row["Stability (ARI)"],
        "Clusters": row["Clusters Found"],
        "Outlier \\%": row["Outlier %"],
    })

latex_df = pd.DataFrame(latex_rows)

# Bold best values
# Best silhouette (highest)
sils = [r["Silhouette"] for r in comparison_rows]
best_sil_idx = np.argmax(sils)
latex_df.iloc[best_sil_idx, 2] = r"\textbf{" + latex_df.iloc[best_sil_idx, 2] + "}"

# Best CH (highest)
chs = [r["Calinski-Harabasz"] for r in comparison_rows]
best_ch_idx = np.argmax(chs)
latex_df.iloc[best_ch_idx, 3] = r"\textbf{" + latex_df.iloc[best_ch_idx, 3] + "}"

# Best DB (lowest)
dbs = [r["Davies-Bouldin"] for r in comparison_rows]
best_db_idx = np.argmin(dbs)
latex_df.iloc[best_db_idx, 4] = r"\textbf{" + latex_df.iloc[best_db_idx, 4] + "}"

latex_str = latex_df.to_latex(
    index=False, escape=False, column_format="llcccccc",
    caption="Clustering method comparison on EdNet KT2 learner features.",
    label="tab:clustering_comparison",
)

print(latex_str)

with open(RESULTS_DIR / "clustering_comparison.tex", "w", encoding="utf-8") as f:
    f.write(latex_str)

print(f"Saved: {RESULTS_DIR / 'clustering_comparison.tex'}")

## 8. t-SNE Visualization (3 panels)

In [ ]:
"""Cell 9 — t-SNE embedding (computed once, reused for all panels)."""

set_global_seed(42)
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_2d = tsne.fit_transform(X)
print(f"t-SNE embedding shape: {X_2d.shape}")

In [ ]:
"""Cell 10 — t-SNE panels: K-Means, GMM, HDBSCAN side by side."""

cluster_cmap = plt.cm.Set2
outlier_color = "#cccccc"

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# --- Panel (a): K-Means ---
ax = axes[0]
km_labels = kmeans_results[best_k_kmeans]["labels"]
for cid in range(best_k_kmeans):
    mask = km_labels == cid
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=[cluster_cmap(cid / max(best_k_kmeans - 1, 1))],
               s=12, alpha=0.6, label=f"C{cid} (n={mask.sum()})")
ax.set_title(f"(a) K-Means (K={best_k_kmeans})", fontsize=13, fontweight="bold")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(fontsize=8, markerscale=2, loc="best")
ax.grid(alpha=0.2)

# --- Panel (b): GMM ---
ax = axes[1]
gmm_labels = gmm_results[best_k_gmm]["labels"]
for cid in range(best_k_gmm):
    mask = gmm_labels == cid
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=[cluster_cmap(cid / max(best_k_gmm - 1, 1))],
               s=12, alpha=0.6, label=f"C{cid} (n={mask.sum()})")
ax.set_title(f"(b) GMM (K={best_k_gmm})", fontsize=13, fontweight="bold")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(fontsize=8, markerscale=2, loc="best")
ax.grid(alpha=0.2)

# --- Panel (c): HDBSCAN ---
ax = axes[2]
hdb_labels = hdbscan_results[best_hdb_key]["labels"]
unique_labels = sorted(set(hdb_labels))
for cid in unique_labels:
    mask = hdb_labels == cid
    if cid == -1:
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=outlier_color, s=6, alpha=0.3,
                   label=f"Outlier (n={mask.sum()})")
    else:
        n_real = len([c for c in unique_labels if c != -1])
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=[cluster_cmap(cid / max(n_real - 1, 1))],
                   s=12, alpha=0.6, label=f"C{cid} (n={mask.sum()})")

hdb_cfg = hdbscan_results[best_hdb_key]
ax.set_title(
    f"(c) HDBSCAN (mcs={hdb_cfg['min_cluster_size']}, ms={hdb_cfg['min_samples']})",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(fontsize=8, markerscale=2, loc="best")
ax.grid(alpha=0.2)

plt.suptitle("t-SNE Visualization of Clustering Methods",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "tsne_comparison.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "tsne_comparison.pdf", bbox_inches="tight")
plt.show()
print("Saved: tsne_comparison.png / .pdf")

## 9. Metric Comparison Bar Charts

In [ ]:
"""Cell 11 — Metric comparison bar charts."""

methods = ["K-Means", "GMM", "HDBSCAN"]
method_colors = ["#3498db", "#2ecc71", "#e74c3c"]

sil_vals = [km_best["silhouette"], gmm_best["silhouette"], hdb_best["silhouette"]]
ch_vals = [km_best["calinski_harabasz"], gmm_best["calinski_harabasz"],
           hdb_best["calinski_harabasz"]]
db_vals = [km_best["davies_bouldin"], gmm_best["davies_bouldin"],
           hdb_best["davies_bouldin"]]
ari_vals = [km_ari_mean, gmm_ari_mean, hdb_ari_mean]
ari_errs = [km_ari_std, gmm_ari_std, hdb_ari_std]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# (a) Silhouette Score
ax = axes[0]
bars = ax.bar(methods, sil_vals, color=method_colors, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, sil_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.005,
            f"{val:.4f}", ha="center", fontsize=9)
ax.set_ylabel("Silhouette Score")
ax.set_title("(a) Silhouette Score (higher=better)", fontweight="bold")
ax.set_ylim(0, max(sil_vals) * 1.25)
sns.despine(ax=ax)

# (b) Calinski-Harabasz
ax = axes[1]
bars = ax.bar(methods, ch_vals, color=method_colors, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, ch_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + max(ch_vals) * 0.01,
            f"{val:.0f}", ha="center", fontsize=9)
ax.set_ylabel("Calinski-Harabasz Index")
ax.set_title("(b) Calinski-Harabasz (higher=better)", fontweight="bold")
ax.set_ylim(0, max(ch_vals) * 1.2)
sns.despine(ax=ax)

# (c) Davies-Bouldin
ax = axes[2]
bars = ax.bar(methods, db_vals, color=method_colors, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, db_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + max(db_vals) * 0.02,
            f"{val:.4f}", ha="center", fontsize=9)
ax.set_ylabel("Davies-Bouldin Index")
ax.set_title("(c) Davies-Bouldin (lower=better)", fontweight="bold")
ax.set_ylim(0, max(db_vals) * 1.3)
sns.despine(ax=ax)

# (d) Stability (ARI)
ax = axes[3]
bars = ax.bar(methods, ari_vals, yerr=ari_errs, color=method_colors,
              edgecolor="black", linewidth=0.5, capsize=5)
for bar, val, err in zip(bars, ari_vals, ari_errs):
    ax.text(bar.get_x() + bar.get_width() / 2, val + err + 0.02,
            f"{val:.4f}", ha="center", fontsize=9)
ax.set_ylabel("Adjusted Rand Index")
ax.set_title("(d) Stability ARI (higher=better)", fontweight="bold")
ax.set_ylim(0, 1.15)
sns.despine(ax=ax)

plt.suptitle("Clustering Quality Metrics Comparison",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "metric_comparison.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "metric_comparison.pdf", bbox_inches="tight")
plt.show()
print("Saved: metric_comparison.png / .pdf")

## 10. Cluster Size Distribution

In [ ]:
"""Cell 12 — Cluster size distribution per method."""

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# K-Means
ax = axes[0]
km_counts = pd.Series(km_labels).value_counts().sort_index()
km_counts.plot.bar(ax=ax, color="#3498db", edgecolor="black", linewidth=0.5)
ax.set_title(f"(a) K-Means (K={best_k_kmeans})", fontweight="bold")
ax.set_xlabel("Cluster")
ax.set_ylabel("Number of Users")
for i, (idx, val) in enumerate(km_counts.items()):
    ax.text(i, val + len(X) * 0.005, str(val), ha="center", fontsize=9)
ax.tick_params(axis="x", rotation=0)
sns.despine(ax=ax)

# GMM
ax = axes[1]
gmm_counts = pd.Series(gmm_labels).value_counts().sort_index()
gmm_counts.plot.bar(ax=ax, color="#2ecc71", edgecolor="black", linewidth=0.5)
ax.set_title(f"(b) GMM (K={best_k_gmm})", fontweight="bold")
ax.set_xlabel("Cluster")
ax.set_ylabel("Number of Users")
for i, (idx, val) in enumerate(gmm_counts.items()):
    ax.text(i, val + len(X) * 0.005, str(val), ha="center", fontsize=9)
ax.tick_params(axis="x", rotation=0)
sns.despine(ax=ax)

# HDBSCAN
ax = axes[2]
hdb_counts = pd.Series(hdb_labels).value_counts().sort_index()
hdb_bar_colors = [outlier_color if idx == -1 else "#e74c3c" for idx in hdb_counts.index]
hdb_counts.plot.bar(ax=ax, color=hdb_bar_colors, edgecolor="black", linewidth=0.5)
ax.set_title(
    f"(c) HDBSCAN (mcs={hdb_cfg['min_cluster_size']}, ms={hdb_cfg['min_samples']})",
    fontweight="bold",
)
ax.set_xlabel("Cluster (-1 = outliers)")
ax.set_ylabel("Number of Users")
for i, (idx, val) in enumerate(hdb_counts.items()):
    ax.text(i, val + len(X) * 0.005, str(val), ha="center", fontsize=9)
ax.tick_params(axis="x", rotation=0)
sns.despine(ax=ax)

plt.suptitle("Cluster Size Distribution",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cluster_sizes.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "cluster_sizes.pdf", bbox_inches="tight")
plt.show()
print("Saved: cluster_sizes.png / .pdf")

## 11. Feature Importance per Cluster (Heatmaps)

In [ ]:
"""Cell 13 — Feature importance heatmap: cluster centroids (z-scored)."""

def compute_cluster_profiles(X, labels, feature_names):
    """Compute mean z-scored feature values per cluster."""
    df = pd.DataFrame(X, columns=feature_names)
    df["cluster"] = labels
    # Exclude outliers
    df = df[df["cluster"] != -1]
    profiles = df.groupby("cluster")[feature_names].mean()
    return profiles


fig, axes = plt.subplots(1, 3, figsize=(24, 7))

titles = [
    f"(a) K-Means (K={best_k_kmeans})",
    f"(b) GMM (K={best_k_gmm})",
    f"(c) HDBSCAN (mcs={hdb_cfg['min_cluster_size']}, ms={hdb_cfg['min_samples']})",
]
all_labels = [km_labels, gmm_labels, hdb_labels]

for ax, title, labels in zip(axes, titles, all_labels):
    profiles = compute_cluster_profiles(X, labels, feature_cols)
    if profiles.empty:
        ax.set_title(title)
        ax.text(0.5, 0.5, "No valid clusters", ha="center", va="center",
                transform=ax.transAxes)
        continue

    sns.heatmap(
        profiles.T, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
        linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8},
        xticklabels=[f"C{c}" for c in profiles.index],
    )
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Feature (z-scored)")
    ax.tick_params(axis="y", rotation=0)

plt.suptitle("Feature Importance per Cluster (Standardized Centroids)",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "feature_heatmap.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "feature_heatmap.pdf", bbox_inches="tight")
plt.show()
print("Saved: feature_heatmap.png / .pdf")

## 12. GMM BIC / AIC Curves

In [ ]:
"""Cell 14 — GMM model selection: BIC and AIC vs K."""

ks = sorted(gmm_results.keys())
bics = [gmm_results[k]["bic"] for k in ks]
aics = [gmm_results[k]["aic"] for k in ks]
gmm_sils = [gmm_results[k]["silhouette"] for k in ks]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BIC / AIC
ax = axes[0]
ax.plot(ks, bics, "o-", color="#e74c3c", linewidth=2, markersize=8, label="BIC")
ax.plot(ks, aics, "s--", color="#3498db", linewidth=2, markersize=8, label="AIC")
ax.axvline(best_k_gmm, color="gray", linestyle=":", alpha=0.6,
           label=f"Best K={best_k_gmm}")
ax.set_xlabel("Number of Components (K)")
ax.set_ylabel("Information Criterion")
ax.set_title("(a) GMM: BIC and AIC vs K", fontweight="bold")
ax.set_xticks(ks)
ax.legend()
sns.despine(ax=ax)

# Silhouette comparison: K-Means vs GMM
ax = axes[1]
km_sils = [kmeans_results[k]["silhouette"] for k in ks]
ax.plot(ks, km_sils, "o-", color="#3498db", linewidth=2, markersize=8, label="K-Means")
ax.plot(ks, gmm_sils, "s--", color="#2ecc71", linewidth=2, markersize=8, label="GMM")
ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("Silhouette Score")
ax.set_title("(b) Silhouette: K-Means vs GMM", fontweight="bold")
ax.set_xticks(ks)
ax.legend()
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "gmm_selection.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "gmm_selection.pdf", bbox_inches="tight")
plt.show()
print("Saved: gmm_selection.png / .pdf")

## 13. HDBSCAN Hyperparameter Grid

In [ ]:
"""Cell 15 — HDBSCAN hyperparameter grid summary."""

hdb_grid_rows = []
for key, res in hdbscan_results.items():
    hdb_grid_rows.append({
        "min_cluster_size": res["min_cluster_size"],
        "min_samples": res["min_samples"],
        "Clusters": res["n_clusters"],
        "Outliers": res["n_outliers"],
        "Outlier %": f"{res['outlier_fraction']:.1%}",
        "Silhouette": f"{res['silhouette']:.4f}" if res["silhouette"] > -1 else "n/a",
        "CH": f"{res['calinski_harabasz']:.0f}" if res["calinski_harabasz"] > 0 else "n/a",
        "DB": f"{res['davies_bouldin']:.4f}" if res["davies_bouldin"] < float('inf') else "n/a",
    })

hdb_grid_df = pd.DataFrame(hdb_grid_rows)
print("HDBSCAN Hyperparameter Grid:")
display(hdb_grid_df)

hdb_grid_df.to_csv(RESULTS_DIR / "hdbscan_grid.csv", index=False)
print(f"\nSaved: {RESULTS_DIR / 'hdbscan_grid.csv'}")

## 14. Agreement Between Methods

In [ ]:
"""Cell 16 — Pairwise ARI between best configurations of each method."""

method_labels = {
    "K-Means": km_labels,
    "GMM": gmm_labels,
    "HDBSCAN": hdb_labels,
}

# For HDBSCAN, only compare non-outlier points
hdb_mask = hdb_labels != -1

print("Pairwise ARI between methods (best configs):")
print("-" * 50)

ari_matrix = np.ones((3, 3))
method_names = list(method_labels.keys())

for i in range(3):
    for j in range(i + 1, 3):
        l1 = method_labels[method_names[i]]
        l2 = method_labels[method_names[j]]

        # If HDBSCAN is involved, restrict to non-outlier points
        if method_names[i] == "HDBSCAN" or method_names[j] == "HDBSCAN":
            mask = hdb_mask
            ari = adjusted_rand_score(l1[mask], l2[mask])
        else:
            ari = adjusted_rand_score(l1, l2)

        ari_matrix[i, j] = ari
        ari_matrix[j, i] = ari
        print(f"  {method_names[i]} vs {method_names[j]}: ARI = {ari:.4f}")

# Heatmap
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    pd.DataFrame(ari_matrix, index=method_names, columns=method_names),
    annot=True, fmt=".4f", cmap="YlOrRd", vmin=0, vmax=1,
    square=True, linewidths=1, ax=ax,
)
ax.set_title("Pairwise ARI Between Methods", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "method_agreement.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "method_agreement.pdf", bbox_inches="tight")
plt.show()
print("Saved: method_agreement.png / .pdf")

## 15. Save All Results

In [ ]:
"""Cell 17 — Save consolidated results."""

results_summary = {
    "experiment": "MARS Clustering Method Comparison (Prompt 3.2)",
    "seeds": SEEDS,
    "n_users": int(X.shape[0]),
    "n_features": int(X.shape[1]),
    "feature_names": feature_cols,
    "kmeans": {
        "best_k": int(best_k_kmeans),
        "silhouette": float(km_best["silhouette"]),
        "calinski_harabasz": float(km_best["calinski_harabasz"]),
        "davies_bouldin": float(km_best["davies_bouldin"]),
        "stability_ari_mean": float(km_ari_mean),
        "stability_ari_std": float(km_ari_std),
        "all_silhouettes": {int(k): float(v["silhouette"])
                            for k, v in kmeans_results.items()},
    },
    "gmm": {
        "best_k": int(best_k_gmm),
        "silhouette": float(gmm_best["silhouette"]),
        "calinski_harabasz": float(gmm_best["calinski_harabasz"]),
        "davies_bouldin": float(gmm_best["davies_bouldin"]),
        "bic": float(gmm_best["bic"]),
        "aic": float(gmm_best["aic"]),
        "stability_ari_mean": float(gmm_ari_mean),
        "stability_ari_std": float(gmm_ari_std),
        "all_bics": {int(k): float(v["bic"])
                     for k, v in gmm_results.items()},
    },
    "hdbscan": {
        "best_config": best_hdb_key,
        "min_cluster_size": int(hdb_best["min_cluster_size"]),
        "min_samples": int(hdb_best["min_samples"]),
        "n_clusters": int(hdb_best["n_clusters"]),
        "outlier_fraction": float(hdb_best["outlier_fraction"]),
        "silhouette": float(hdb_best["silhouette"]),
        "calinski_harabasz": float(hdb_best["calinski_harabasz"]),
        "davies_bouldin": float(hdb_best["davies_bouldin"]),
        "stability_ari_mean": float(hdb_ari_mean),
        "stability_ari_std": float(hdb_ari_std),
    },
    "output_files": [
        str(RESULTS_DIR / "clustering_comparison.csv"),
        str(RESULTS_DIR / "clustering_comparison.tex"),
        str(RESULTS_DIR / "hdbscan_grid.csv"),
        str(RESULTS_DIR / "tsne_comparison.png"),
        str(RESULTS_DIR / "metric_comparison.png"),
        str(RESULTS_DIR / "cluster_sizes.png"),
        str(RESULTS_DIR / "feature_heatmap.png"),
        str(RESULTS_DIR / "gmm_selection.png"),
        str(RESULTS_DIR / "method_agreement.png"),
    ],
}

with open(RESULTS_DIR / "clustering_comparison_meta.json", "w") as f:
    json.dump(results_summary, f, indent=2, default=str)

print("All results saved to:", RESULTS_DIR)
for fp in results_summary["output_files"]:
    print(f"  {fp}")

## Key Findings (expected)

1. **K-Means and GMM produce similar clusterings** -- ARI between the two should be
   high (>0.8), confirming the data is roughly spherical and Gaussian assumptions hold.
2. **K-Means is preferred** -- comparable quality to GMM but faster, simpler, and more
   stable across seeds (higher ARI stability).
3. **HDBSCAN finds interesting outlier clusters** -- identifies atypical learner profiles
   (e.g., very fast guessers, extremely slow methodical students), but the outlier
   fraction may be too high for production use.
4. **Silhouette and CH scores** align in ranking K-Means >= GMM > HDBSCAN.
5. **Davies-Bouldin** is lowest for K-Means, confirming well-separated clusters.
6. **Conclusion:** K-Means is the right default for MARS PersonalizationAgent.
   HDBSCAN outlier detection could be added as a post-processing step.